<a href="https://colab.research.google.com/github/saathvikMD/dqn/blob/main/lunar_lander_engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
#remove " > /dev/null 2>&1" to see what is going on under the hood
!pip install gym pyvirtualdisplay > /dev/null 2>&1
!apt-get install -y xvfb python-opengl ffmpeg > /dev/null 2>&1

In [5]:
!apt-get update > /dev/null 2>&1
!apt-get install cmake > /dev/null 2>&1
!pip install --upgrade setuptools 2>&1
!pip install ez_setup > /dev/null 2>&1
!pip install gym[atari] > /dev/null 2>&1

     |████████████████████████████████| 2.0MB 4.3MB/s 
ERROR: datascience 0.10.6 has requirement folium==0.2.1, but you'll have folium 0.8.3 which is incompatible.
  Found existing installation: setuptools 51.0.0
    Uninstalling setuptools-51.0.0:
      Successfully uninstalled setuptools-51.0.0


In [6]:
!pip install box2d.py
!pip install gym[Box_2D]

     |████████████████████████████████| 450kB 6.1MB/s 


In [7]:
import gym
from gym import logger as gymlogger
from gym.wrappers import Monitor
gymlogger.set_level(40) #error only
import tensorflow as tf
import numpy as np
import random
import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline
import math
import glob
import io
import base64
from IPython.display import HTML

from IPython import display as ipythondisplay

In [8]:
from pyvirtualdisplay import Display
display = Display(visible=0, size=(1400, 900))
display.start()

In [9]:
"""
Utility functions to enable video recording of gym environment and displaying it
To enable video, just do "env = wrap_env(env)""
"""

def show_video():
  mp4list = glob.glob('video/*.mp4')
  if len(mp4list) > 0:
    mp4 = mp4list[0]
    video = io.open(mp4, 'r+b').read()
    encoded = base64.b64encode(video)
    ipythondisplay.display(HTML(data='''<video alt="test" autoplay 
                loop controls style="height: 400px;">
                <source src="data:video/mp4;base64,{0}" type="video/mp4" />
             </video>'''.format(encoded.decode('ascii'))))
  else: 
    print("Could not find video")
    

def wrap_env(env):
  env = Monitor(env, './video', video_callable=lambda episode_id: episode_id%1==0, force=True)
  return env

In [10]:
import gym
import keras
import random
import numpy as np

In [11]:
class Engine():
  def __init__(self, max_memory, env_name, lr):
    self.lr = lr
    self.memory = []
    self.discount = 0.9
    self.max_memory = max_memory
    self.env_name = env_name
    self.env = wrap_env(gym.make(self.env_name))
    self.epsilon = 0.99
    self.epsilon_decay_rate = 0.05
    self.epsilon_deacy_rate_decay = 0.009
    self.min_epsilon = 0.8
    self.input_shape = len(self.env.reset())
    self.output_shape = self.env.action_space.n
    self.model = keras.models.Sequential()
    self.model.add(keras.layers.Dense(self.input_shape, input_shape = (1, self.input_shape)))
    self.model.add(keras.layers.Dense(self.input_shape * 2))
    self.model.add(keras.layers.Dense(self.input_shape * 3))
    self.model.add(keras.layers.Dense(self.input_shape * 4))
    self.model.add(keras.layers.Dense(self.input_shape * 3))
    self.model.add(keras.layers.Dense(self.output_shape * 2))
    self.model.add(keras.layers.Dense(self.output_shape))
    self.model.compile(loss = 'mean_squared_error', optimizer = keras.optimizers.Adam(learning_rate=self.lr))
  
  def remember(self, current_state, action, reward, next_state, game_over):
      transition = [current_state, action, reward, next_state]
      self.memory.append([transition, game_over])

  def get_batch(self, batch_size, model):
      len_memory = len(self.memory)
      num_inputs = self.input_shape
      num_outputs = self.output_shape

      inputs = np.zeros((min(batch_size, len_memory), num_inputs))
      targets = np.zeros((min(batch_size, len_memory), num_outputs))
      for i, inx in enumerate(np.random.randint(0, len_memory, size = min(batch_size, len_memory))):
          current_state, action, reward, next_state = self.memory[inx][0]
          game_over = self.memory[inx][1]

          inputs[i] = current_state
          targets[i] = model.predict(np.array(current_state).reshape(1, 1, num_inputs))[0][0]

          if game_over:
              targets[i][action] = reward
          else:
              targets[i][action] = np.argmax(reward + self.discount * model.predict(np.array(next_state).reshape(1, 1, num_inputs))[0][0])

      return inputs, targets

  def train(self, training_epochs = 100, target_change = None):
    if target_change == None:
      target_change = training_epochs/20
    target_model = keras.models.clone_model(self.model)
    rewards = []
    j = 0
    for i in range(training_epochs):
      j = 0
      game_over = False
      epsilon = self.epsilon
      self.env.reset()
      current_state = np.zeros(self.input_shape).reshape(1, 1, self.input_shape)
      total_reward = 0
      k = 0
      while not game_over:
        k += 1
        j += 1
        if j == target_change:
          target_model = self.model
          j = 0 
        if random.random() > epsilon:
          action = random.randint(0, 1)
        else:
          action = np.argmax(self.model.predict(current_state))
        next_state, reward, game_over, info = self.env.step(action)
        next_state = next_state.reshape(1, 1, self.input_shape)
        self.remember(current_state, action, reward, next_state, game_over)
        inputs, outputs = self.get_batch(20, target_model)
        self.model.fit(inputs, outputs, epochs = 1, verbose = 0)

        total_reward += reward

        epsilon = epsilon - self.epsilon_decay_rate
        self.epsilon_decay_rate = self.epsilon_decay_rate + self.epsilon_deacy_rate_decay
        current_state = next_state
        print('\rEpisode ' + str(i)+ ' - total reward:' + str(total_reward), 'moves:' + str(k), end = '')
      rewards.append(total_reward)
      print('\rEpisode ' + str(i)+ ' - total reward:' + str(total_reward))
    plt.plot(rewards)
    plt.show()

In [12]:
engine = Engine(2000, 'LunarLander-v2', 0.001)
engine.train(training_epochs = 100, target_change=100)